In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from snsphd import viz
import math
import orjson

Colors, pallet = viz.phd_style(
    jupyterStyle=True, grid=True, axese_width=0.5, text=6, svg_mode=True
)

from enum import Enum
from dataclasses import dataclass, fields, field, _MISSING_TYPE

from load_schema import Decode, Result, Event

import matplotlib as mpl
import os

# set the DPI for all figures
mpl.rcParams["figure.dpi"] = 100

from snsphd.layout import bisect

from numba import njit
from snsphd.viz import save_bokeh_dark_json
from snsphd.layout import bisect

from pydantic import BaseModel

# from overlap_measure import BiGauss

import networkx as nx

from matplotlib.colors import LogNorm
from scipy.signal import fftconvolve
import cv2 as cv

In [ ]:
from worker import DecodingResults


# this takes a few seconds
with open("../inter/output.json", "rb") as file:
    dic = orjson.loads(file.read())

    dec = DecodingResults(**dic)


with open("../inter/decode_20GHz26.0.json", "r") as f:
    data = orjson.loads(f.read())
    decode = Decode(**data)

db_values = np.arange(16, 50, 2).tolist()
print(db_values)


# need type hints for intellisense to work
def decode_from_folder(folder_path: str, db_values: list[int]) -> list[Decode]:
    decodes = []
    for db in db_values:
        filename = f"decode_20GHz{db}.0.json"
        filepath = os.path.join(folder_path, filename)
        with open(filepath, "r") as f:
            data = orjson.loads(f.read())
            decode = Decode(**data)
            decodes.append(decode)
    return decodes


decodes = decode_from_folder("../inter", db_values)

In [ ]:
def calculate_mfr(struct: list[Decode]) -> list[float]:
    mfrs = []
    for d in range(len(struct)):
        missing = 0
        found = 0
        for res in struct[d].results[0]:
            result: Result = res.result
            if result == Result.MISSING:
                missing += 1
            if (result == Result.CORRECT) or (result == Result.INCORRECT):
                found += 1
        total = missing + found

        ratio_of_vacuum = missing / total
        mfr = -np.log(ratio_of_vacuum)
        mfrs.append(mfr)

    return mfrs


mfr = calculate_mfr(decodes)

mfr = np.flip(mfr)
# plt.plot(mfr)

ln_mfr = np.log(mfr)
x = np.arange(len(ln_mfr))
slope, intercept = np.polyfit(x[:-3], ln_mfr[:-3], 1)
ln_mfr_ext = slope * x + intercept


ln_mfr_corrected = ln_mfr.copy()
ln_mfr_corrected[-4:] = ln_mfr_ext[-4:]
mfr_corrected = np.exp(ln_mfr_corrected)


plt.plot(x, mfr_corrected, "o", label="corrected mean photon rate")
plt.plot(x, mfr, "o", label="measured mean photon rate")
plt.yscale("log")
plt.legend()

# put the order back to what it was
mfr = np.flip(mfr)
mfr_corrected = np.flip(mfr_corrected)

In [ ]:
from load_schema import GMData, GMDataPhotonCommunity
from matplotlib.patches import Ellipse
import load_schema


def draw_ellipse(position, covariance, ax=None, single_ellipse=False, **kwargs):
    """Draw an ellipse with a given position and covariance"""
    ax = ax or plt.gca()

    # Convert covariance to principal axes
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)

    # Draw the Ellipse
    if single_ellipse:
        nsig = 3
        ax.add_patch(
            Ellipse(
                position, nsig * width, nsig * height, angle=angle, fill=False, **kwargs
            )
        )
    else:
        for nsig in np.linspace(0, 4, 8):
            ax.add_patch(
                Ellipse(
                    position,
                    nsig * width,
                    nsig * height,
                    angle=angle,
                    fill=False,
                    **kwargs
                )
            )


def plot_gmm(
    ax,
    g: GMData | GMDataPhotonCommunity,
    label=True,
    data_alpha=0.2,
    single_ellipse=False,
    **ellipse_kwargs
):
    """for plotting a all ellipses in a GMdata structure. single_ellipse controlls if one ellipse or more is plotted for each gaussian"""
    # w_factor = 0.2 / g.weights.max()

    if isinstance(g, GMData):
        for pos, covar, w in zip(g.means, g.covariances, g.weights):
            draw_ellipse(pos, covar, ax=ax, single_ellipse=single_ellipse, **ellipse_kwargs)
    elif isinstance(g, GMDataPhotonCommunity) or isinstance(g, load_schema.GMDataPhotonCommunity):
        for pos, covar, w in zip(g.photon_means, g.photon_covariances, g.photon_weights):
            draw_ellipse(pos, covar, ax=ax, single_ellipse=single_ellipse, **ellipse_kwargs)
    else:
        print(type(g))
        raise TypeError("g must be of type GMData or GMDataPhotonCommunity")




def plot_gmm_single(
    ax,
    gm_data: GMData,
    ellipse_idx=0,
    label=True,
    data_alpha=0.2,
    single_ellipse=False,
    **ellipse_kwargs
):
    """for plotting a single ellipse in a GMdata structure. single_ellipse controlls if one ellipse or more is plotted for each gaussian"""
    w_factor = 0.2 / gm_data.weights.max()

    draw_ellipse(
        gm_data.means[ellipse_idx],
        gm_data.covariances[ellipse_idx],
        ax=ax,
        single_ellipse=single_ellipse,
        **ellipse_kwargs
    )

In [ ]:
import copy

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

dbidx = 8

comp_idx = 10

st = copy.deepcopy(decodes[dbidx].gm_data.gm_list[comp_idx])
print(f"shifting means x by { np.average(st.means[:,0], weights=st.weights)}")
print(f"shifting means y by { np.average(st.means[:,1], weights=st.weights)}")


st.means[:, 0] = st.means[:, 0] - np.average(st.means[:, 0], weights=st.weights)
st.means[:, 1] = st.means[:, 1] - np.average(st.means[:, 1], weights=st.weights)


plot_gmm(ax, st, single_ellipse=True, lw=1.5, alpha=0.8)

plot_gmm_single(ax, st, ellipse_idx=5, single_ellipse=True, lw=3, alpha=1, color="red")


# for pos, w in zip(decodes[dbidx].gm_data.gm_list[comp_idx].means, decodes[dbidx].gm_data.gm_list[comp_idx].weights):
#     print("position: ", pos, "weight: ", w)

ax.set_xlim(-500, 500)
ax.set_ylim(-500, 500)


# you will make sub-GMDatas for each distinct photon number
# then, for each even, I guess you test all sub-gms and assign the event a photon number based on most probable.
# I guess you could arrange the sub-gms in order of most to least probable.

# how is photon number confidence determined?

In [ ]:
# from overlap_measure import BiGauss
from kl_divergence import kl_mvn

st = copy.deepcopy(decodes[dbidx].gm_data.gm_list[comp_idx])

st.means[:, 0] = st.means[:, 0] - np.average(st.means[:, 0], weights=st.weights)
st.means[:, 1] = st.means[:, 1] - np.average(st.means[:, 1], weights=st.weights)

# loc = decodes[dbidx].gm_data.gm_list[comp_idx]

size = len(st.means)

adj_matrix = np.zeros((size, size), dtype=np.float64)
adj_matrix_kl = np.zeros((size, size), dtype=np.float64)

for i, (pos, cov, w) in enumerate(zip(st.means, st.covariances, st.weights)):
    pos_ = pos / 100  # scale down position

    cov_ = cov / 10000  # scale down covariance
    w_ = w
    for j, (pos_inner, cov_inner, w_inner) in enumerate(
        zip(st.means, st.covariances, st.weights)
    ):
        if i == j:
            continue
        pos_inner_ = pos_inner / 100  # scale down position
        cov_inner_ = cov_inner / 10000  # scale down covariance
        w_inner_ = w_inner

        adj_matrix_kl[i, j] = kl_mvn((pos_inner_, cov_inner_), (pos_, cov_)) + kl_mvn(
            (pos_, cov_), (pos_inner_, cov_inner_)
        )  # +

adj_kl_log_distance = (1 - np.log(adj_matrix_kl)) - np.min((1 - np.log(adj_matrix_kl)))

In [ ]:
arr = 40 - adj_matrix_kl

arr[arr < 0] = 0
arr = arr / np.max(arr)

In [ ]:
for q in range(len(adj_kl_log_distance)):
    adj_kl_log_distance[q, q] = 0

adj_kl_log_distance = adj_kl_log_distance / np.max(adj_kl_log_distance)

In [ ]:
from scipy.stats import multivariate_normal

st = copy.deepcopy(decodes[dbidx].gm_data.gm_list[comp_idx])

st.means[:, 0] = st.means[:, 0] - np.average(st.means[:, 0], weights=st.weights)
st.means[:, 1] = st.means[:, 1] - np.average(st.means[:, 1], weights=st.weights)


def total_multivariate_normal(gm_data: GMData, X):
    gausses = []
    total = 0
    for pos, cov, w in zip(gm_data.means, gm_data.covariances, gm_data.weights):
        gausses.append(multivariate_normal(mean=pos, cov=cov, allow_singular=True))

    for w, mean, gauss in zip(gm_data.weights, gm_data.means, gausses):
        total += w * (gauss.pdf(X) / gauss.pdf(mean))

    return total


x, y = np.mgrid[-300:300:3, -300:300:1]
pos = np.dstack((x, y))
plt.contourf(x, y, total_multivariate_normal(st, pos))

In [ ]:
G = nx.from_numpy_array(arr)
# print(G.edges(data=True))
res = nx.community.louvain_communities(G, resolution=0.4)

# red = nx.community.k_clique_communities(G, 1)
res

In [ ]:
from util import darken_and_saturate

# Test the function
nice_hex_colors = [
    "#FF9191",
    "#9391FF",
    "#FFCC91",
    "#DC91FF",
    "#91FF9C",
    "#FF9191",
    "#9391FF",
    "#FFCC91",
    "#DC91FF",
]
dark_saturated_colors = [darken_and_saturate(color) for color in nice_hex_colors]
print(dark_saturated_colors)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

plot_gmm(ax, st, single_ellipse=True, lw=1.5, alpha=0.8)


x, y = np.mgrid[-300:300:3, -300:300:1]
pos = np.dstack((x, y))
ax.contourf(x, y, total_multivariate_normal(st, pos), alpha=1, cmap="gray_r")

colors = ["red", "blue", "green", "purple", "orange", "pink", "cyan", "magenta"]

cl = dark_saturated_colors[: len(res)]
for color, ress in zip(cl, res):
    for item in ress:
        plot_gmm_single(
            ax,
            st,
            ellipse_idx=item,
            single_ellipse=True,
            lw=1.5,
            alpha=0.7,
            color=color,
        )
        # print(st.weights[item])


ax.set_xlim(-250, 250)
ax.set_ylim(-250, 250)
viz.save_light_dark_all(fig, "../figs/thing")

In [ ]:
np.average(
    decodes[dbidx].gm_data.gm_list[comp_idx].means,
    weights=decodes[dbidx].gm_data.gm_list[comp_idx].weights,
    axis=0,
)

In [ ]:
from matplotlib import cm
import matplotlib

# something that takes in res
from util import remove_far_field_gaussians, filter_unwanted_gaussians, calculate_photon_communities, photon_number_bifurcate_gm_data


# for each gaussian in set of res, create data structure. Basically just GMData instances


# you need to create a data structure for holding multiple GMdata instances indexed by photon number


fig, axs = plt.subplots(3, 9, figsize=(19, 8))
splitting = np.linspace(0.7, 0.2, 10)
print(splitting)
dbidxs = np.arange(0, len(splitting) * 2, 2)

print(len(splitting))
print(len(dbidxs))
for i, (dbidx, splitter) in enumerate(zip(range(0, 18, 2), splitting)):
    # print("dbidx: ", dbidx)

    for j, component_index in enumerate(range(6, 15, 3)):
        ax = axs[j, i]

        st2 = copy.deepcopy(decodes[dbidx].gm_data.gm_list[component_index])
        # st2.means = st2.means- np.average(st2.means, weights=st2.weights)
        if i == 1:
            print("number of components: ", len(st2.means))
        st2.means[:, 0] = st2.means[:, 0] - np.average(
            st2.means[:, 0], weights=st2.weights
        )
        st2.means[:, 1] = st2.means[:, 1] - np.average(
            st2.means[:, 1], weights=st2.weights
        )

        st2 = remove_far_field_gaussians(st2, gaussian_distance_cutoff=500)
        st2 = filter_unwanted_gaussians(
            st2, stretch_ratio_cutoff=10, large_cutoff=2500, small_cutoff=10
        )

        res = calculate_photon_communities(
            st2, far_field_cutoff=40, splitting_control=splitter
        )
        # print(res)
        out = photon_number_bifurcate_gm_data(st2, res)

        viridis = matplotlib.colormaps["viridis"]  # Get the 'viridis' colormap
        for k in range(len(out)):
            color = viridis(k / len(out))  # Get the color from the colormap
            color_2 = dark_saturated_colors[k]
            plot_gmm(
                ax, out[k].gm_photon, color=color_2, lw=2, single_ellipse=True, alpha=0.5
            )
        ax.set_xlim(-350, 350)
        ax.set_ylim(-350, 350)

# you need to tune the splitting parameter depending on the mean photon rate
# and it seems more consistent to get rid of the far field gaussians before computing res. Just remake GMData with only near field gaussians

# viz.save_light_dark_all(fig, "../figs/thing3")

In [ ]:
from util import StaticSimulation, PhotonGMData, poisson

dbidx = 4  # max is 16  # 5 is messed up
component_index = 15

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

st_single = copy.deepcopy(decodes[dbidx].gm_data.gm_list[component_index])
print("number of components: ", len(st_single.means))

st_single = remove_far_field_gaussians(st_single, gaussian_distance_cutoff=1000)
st_single = filter_unwanted_gaussians(
    st_single, stretch_ratio_cutoff=10, large_cutoff=2500, small_cutoff=10
)
res = calculate_photon_communities(
    st_single, far_field_cutoff=40, splitting_control=0.7
)

out: list[PhotonGMData] = photon_number_bifurcate_gm_data(st_single, res)

for k in range(len(out)):
    color = viridis(k / len(out))  # Get the color from the colormap
    color_2 = dark_saturated_colors[k]
    plot_gmm(ax, out[k].gm_photon, color=color_2, lw=2, single_ellipse=True, alpha=0.5)


# create the simulation that will be used for all further photon descrminiation analysis:
static_sim: StaticSimulation = StaticSimulation(out)

ax.set_xlim(-300, 300)
ax.set_ylim(200, 1000)
ax.set_aspect("equal")  # Set the aspect ratio to be equal

In [ ]:
# this is the mean photon number used for constructed the PNR dependent GMM model

print(mfr_corrected[4])

In [ ]:
static_sim.to_json("../inter/static_sim.json")

In [ ]:
test_new_static_sim = StaticSimulation.from_json("../inter/static_sim.json")

In [ ]:
static_sim.single_photon_com

In [ ]:
from util import gaussian_2d


def create_grid_from_gm_data(
    g: GMData | GMDataPhotonCommunity, x_range: tuple[int], y_range: tuple[int], resolution: int = 2
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x, y = np.meshgrid(
        np.arange(x_range[0], x_range[1], resolution), np.arange(y_range[0], y_range[1], resolution)
    )
    return fill_grid_from_gm_data(g, x, y)


def fill_grid_from_gm_data(
    gm_data: GMData | GMDataPhotonCommunity, x, y
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    z = np.zeros(np.shape(x))
    if isinstance(gm_data, GMData):
        # print(type(gm_data))
        for pos, covar, w in zip(gm_data.means, gm_data.covariances, gm_data.weights):
            z = z + gaussian_2d(x, y, covar, [pos[0], pos[1]]) * w
    elif isinstance(gm_data, GMDataPhotonCommunity):
        # print(type(gm_data))
        for pos, covar, w in zip(gm_data.photon_means, gm_data.photon_covariances, gm_data.photon_weights):
            z = z + gaussian_2d(x, y, covar, [pos[0], pos[1]]) * w
    else:
        # print(type(gm_data))
        raise Exception("gm_data must be of type GMData or GMDataPhotonCommunity")
    return x, y, z

In [ ]:
import plotly
import plotly.graph_objects as go


y_range = (300, 900)
x_range = (-200, 200)

x, y = np.meshgrid(
    np.arange(x_range[0], x_range[1], 2), np.arange(y_range[0], y_range[1], 2)
)

x = x.astype(np.int16)
y = y.astype(np.int16)

mfr = 3

# community = out[0].gm_data
colors = ["Greens", "Oranges", "Blues", "Reds", "Greens", "Oranges", "Blues", "Reds"]


mfr_sim_out = np.array(static_sim.scale_simulation_to_match_mfr(mfr))


fig = go.Figure()
for i, community in enumerate(mfr_sim_out):
    z = np.zeros(np.shape(x))
    com = community.gm_photon
    _, _, z = fill_grid_from_gm_data(com, x, y)
    showscale = (
        True if i == 0 else False
    )  # set showscale to True for the first surface only

    z_max = np.max(z)
    z = z + z_max * 0.0005 * i
    z = np.round(z, decimals=6)


    x = x.astype(np.int16)
    y = y.astype(np.int16)
    z = z.astype(np.float32)

    surf = go.Surface(
        colorscale=colors[i],
        contours={
            "x": {
                "show": True,
                "start": x_range[0],
                "end": x_range[1],
                "size": (x_range[1] - x_range[0]) / 40,
                "color": "rgba(0,20,0,12)",
            },
            "y": {
                "show": True,
                "start": y_range[0],
                "end": y_range[1],
                "size": (y_range[1] - y_range[0]) / 40,
                "color": "rgba(0,0,0,12)",
            },
            "z": {
                "show": True,
                "start": 0.000006,
                "end": 0.01,
                "size": 0.0003,
                "color": "rgba(0,0,0,3)",
            },
        },
        x=x,
        y=y,
        z=z,
        showscale=showscale,
    )

    print(f"adding trace {i}")
    fig.add_trace(surf)

fig.update_layout(
    # title="3d plot",
    autosize=True,
    width=800,
    height=700,
    scene_aspectmode="manual",
    scene_aspectratio=dict(x=1, y=1, z=0.2),
    scene=dict(
        xaxis_title=r't_A',
        yaxis_title=r't_B',
        zaxis_title='Amplitude (a.u.)',
        xaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor="black",
        ),
        camera=dict(
            up=dict(x=0, y=0, z=1),
            center=dict(x=0, y=0, z=0),
            eye=dict(x=1, y=1, z=0.02),
        ),
    ),
)

line_marker = dict(color="#101010", width=4)
fig.add_scatter3d(x=x, y=y, z=z + 0.03, mode="lines", line=line_marker, name="")

fig.show()
fig.write_html("custom_groupings.html")
plotly.offline.plot(
    fig, filename="../www/normalized_photon_groupings.html", auto_open=False
)

In [ ]:
from cv2 import matchTemplate
from scipy.ndimage import convolve
from scipy.signal import correlate

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
# decodes goes 0 to 16
for i, dbidx in enumerate(range(len(decodes))):
    # 5 is off,
    # 4 and lower are off in a similar way

    if i == 0:
        print("db value: ", db_values[dbidx])

        color_2 = "black"  # dark_saturated_colors[i]
        component_index = 8
        st2 = copy.deepcopy(decodes[dbidx].gm_data.gm_list[component_index])
        st2 = remove_far_field_gaussians(st2, gaussian_distance_cutoff=1000)
        st2 = filter_unwanted_gaussians(
            st2,
            stretch_ratio_cutoff=20,
            large_cutoff=3000,
            small_cutoff=1,
            weight_cutoff=0.00001,
        )
        res = calculate_photon_communities(
            st2, far_field_cutoff=40, splitting_control=0.1
        )
        out = photon_number_bifurcate_gm_data(st2, res)

        com_1_photon = np.average(
            out[0].gm_photon.photon_means, weights=out[0].gm_photon.photon_weights, axis=0
        )
        origin = static_sim.single_photon_com.tolist()

        correction_x = 0
        correction_y = 0

        if i >= 3:
            correction_x = origin[0] - com_1_photon[0]
            correction_y = origin[1] - com_1_photon[1]
            # print("correction x: ", correction_x)
            # print("correction y: ", correction_y)
            correction_vector = [correction_x, correction_y]

            correction_dict = {"origin": origin, "correction_vector": correction_vector}
            with open(f"../inter/offset_correction_{db_values[dbidx]}.json", "w") as f:
                f.write(json.dumps(correction_dict))

            for thing in out:
                thing.gm_photon.photon_means = thing.gm_photon.photon_means + correction_vector 

        x_range = (-1500, 700)
        y_range = (-800, 1500)
        resolution = 2

        mfr_accurate_sim = static_sim.flatten_sim_at_mfr(mfr_corrected[i])
        sim_x, sim_y, sim_z = create_grid_from_gm_data(mfr_accurate_sim, x_range, y_range, resolution=resolution)
        

        # fudge factors
        if i == 2:
            correction_x += 33 # a little fudge factor
            correction_y += 18

        if i == 1:
            correction_x += 48 # a little fudge factor
            correction_y += 18

        if i == 0:
            correction_x += 64 # a little fudge factor
            correction_y += 18

        if i < 3:
            # in this dB range, the 1 photon events are to rare and can't be used for alignment
            # instead rely on convolutions of the current distribution with the i==3 distribution
            unaligned_x, unaligned_y, unaligned_z = create_grid_from_gm_data(st2, x_range, y_range, resolution=resolution)
            convolved = correlate(unaligned_z, sim_z, mode='same', method='fft')

            max_index = np.argmax(convolved)
            max_coords = np.unravel_index(max_index, convolved.shape)
            max_coords = [max_coords[0] - np.shape(convolved)[0]/2, max_coords[1] - np.shape(convolved)[1]/2]
            correction_x += max_coords[0]*2
            correction_y += max_coords[1]*2
            

            correction_vector =  -np.array([correction_x, correction_y])

            print("correction_vector: ", correction_vector)

            for thing in out:
                thing.gm_photon.photon_means = thing.gm_photon.photon_means + correction_vector

            correction_dict = {"origin": origin, "correction_vector": correction_vector.tolist()}
            with open(f"../inter/offset_correction_{db_values[dbidx]}.json", "w") as f:
                f.write(json.dumps(correction_dict))


        # from here, we just modify the means for viewing purposes. Not modifying data that will be saved
        ax.hist2d(
            decodes[dbidx].gm_data.counts[:, 0] + correction_vector[0],
            decodes[dbidx].gm_data.counts[:, 1] + correction_vector[1],
            bins=np.arange(-1500, 1500, 2),
            norm=LogNorm(),
            rasterized=True,
            alpha=0.5,
        )

        viridis = matplotlib.colormaps["viridis"]  # Get the 'viridis' colormap
        for k in range(len(out)):
            color = viridis(k / len(out))  # Get the color from the colormap
            color_2 = dark_saturated_colors[k]
            plot_gmm(
                ax,
                out[k].gm_photon,
                color=color_2,
                lw=0.5,
                single_ellipse=True,
                alpha=0.5,
            )
        for k in range(len(static_sim.data)):
            color_2 = dark_saturated_colors[k]
            plot_gmm(
                ax,
                static_sim.data[k].gm_photon,
                color=color_2,
                lw=2,
                single_ellipse=True,
                alpha=0.9,
            )

ax.set_xlim(-180, 180)
ax.set_ylim(400, 850)
# set the plot square:
ax.set_aspect("equal")  # Set the aspect ratio to be equal

ax.axvline(x=origin[0], alpha=0.2, color="black")
ax.axhline(y=origin[1], alpha=0.2, color="black")

In [ ]:
# no you should be able to rescan all the data by modifying the function "correction_from_gaussian_model" to support photon number discrimination

In [ ]:
from scipy.signal import convolve
# convolved = correlate(unaligned_z, basis_z, mode='same', method='fft')
# convolution = correlate(basis_z, unaligned_z, mode='same', method='fft')
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# print(np.shape(basis_z))

# print(np.shape(convolved))
# ax.imshow(unaligned_z, origin="lower", extent=[-1200, 400, -500, 1200], cmap="viridis", alpha=0.5)


# # # convolved = correlate(basis_z, unaligned_z, mode='same', method='fft')
# ax.imshow(sim_z, origin="lower", extent=[-1200, 400, -500, 1200], cmap="plasma", alpha=0.5)


# # print(np.shape(basis_z))
# ax.imshow(basis_z, origin="lower", extent=[-1200, 400, -500, 1200], cmap="viridis", alpha=0.5)
# ax.imshow(unaligned_z, origin="lower", extent=[-1200, 400, -500, 1200], cmap="viridis", alpha=0.5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
# convolved[549,741] = 10
ax.imshow(basis_z, origin="lower", extent=[-1200, 400, -500, 1200], cmap="viridis", alpha=0.5)
ax.imshow(unaligned_z, origin="lower", extent=[-1200, 400, -500, 1200], cmap="viridis", alpha=0.5)
ax.imshow(np.flip(convolved), origin="lower", extent=[-1200, 400, -500, 1200], cmap="viridis", alpha=0.5)

# ax.imshow(basis_z)
# print(np.shape(convolved))

In [ ]:
# you have gaussian setups that are good
# you need a way to tune the weights for a given setup to match the other dBs
# I guess you need to integrate over the area of the gaussian group. The integrations should match the relative intensities of poisson stats
# -> no you don't each gaussian group is not normalized to 1. All you need is the weight.


# So for a given data source, you can calculate mu

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
im = ax.imshow(arr, cmap="viridis", interpolation="nearest", origin="upper")

# Create a new axes for the colorbar that is the same size as the original axes
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.28)

fig.colorbar(im, cax=cax)

viz.save_light_dark_all(fig, "../figs/adj_matrix")